Задача 1. Удвоение чисел и получение результата
Подробное описание задачи:
1. У вас есть список списков с числами, которые должны обрабатываться. Пример списка можно загрузить из файла test_list_numbers.txt (находится в материалах к заданию).
Напишите многопоточный код для обработки чисел из нескольких списков. Каждое число в списке должно быть умножено на 2, с имитацией задержки 0.2 сек на каждой операции.
Используйте ThreadPoolExecutor и as_completed для управления потоками и отслеживания результатов.
2. Реализовать функцию process_number(number), которая принимает число, умножает его на 2, имитирует задержку в 0.2 секунды (рекомендуется через time.sleep(0.2)) и возвращает результат.
3. Инициализировать ThreadPoolExecutor с определенным количеством рабочих потоков (рекомендуется, 10). Использовать метод submit() для отправки задачи обработки каждого числа из всех списков в пул потоков. Сохраните возвращаемые объекты Future в списке.
4. Итерируйтесь через объекты Future, используя as_completed(), чтобы получить результаты задач по мере их завершения.
5. После завершения всех задач, выведите сумму обработанных чисел списка который был обработан быстрее остальных. Вывод программы должен быть следующим:
Выход программы 1:
Сумма чисел в первом обработанном списке: 11090



In [7]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# 2. Каждое число умножаем на 2 с задержкой 0.2 сек.
def process_number(number):
    time.sleep(0.2)  # задержка 0.2 сек
    return number * 2

# 1. Загружаем файл 
def load_data(filename):
    with open(filename, "r") as f:
        content = f.read()
    
    # Убираем скобки, переводы строк, лишние пробелы
    content = content.replace('[', '').replace(']', '')
    content = content.replace('\n', ' ').replace('\r', ' ')
    content = ' '.join(content.split())  # убираем лишние пробелы
    
    # Получаем все числа как строки: ['175', '790', '103', ...]
    number_strings = [x for x in content.split(',') if x.strip().isdigit()]
    
    # В int: [175, 790, 103, ...]
    numbers = [int(x) for x in number_strings]
    
    # Разбиваем на списки по размерам из файла 
    list_sizes = [12, 24, 20, 16, 15, 21, 15, 18, 25, 23]
    
    lists_of_numbers = []
    idx = 0
    for size in list_sizes:
        lists_of_numbers.append(numbers[idx:idx + size])
        idx += size
    
    return lists_of_numbers

# 3–5. Создание и работа с потоками ThreadPoolExecutor + as_completed
def main():
    filename = "test_list_numbers.txt"
    lists_of_numbers = load_data(filename)
    
    print(f"Загружено списков: {len(lists_of_numbers)}")
    
    future_to_info = {} # номер списка
    all_futures = [] # все списки
    
    # 3. Создаем потоки
    with ThreadPoolExecutor(max_workers=10) as executor:
        for list_idx, numbers in enumerate(lists_of_numbers):
            for number in numbers:
                future = executor.submit(process_number, number)
                future_to_info[future] = list_idx  # только индекс списка
                all_futures.append(future)
        
        # 4. Результаты по мере завершения
        list_results = {}           # список обработанных чисел
        first_finished_list_idx = None
        
        for future in as_completed(all_futures):
            list_idx = future_to_info[future]
            
            result = future.result()
            
            if list_idx not in list_results:
                list_results[list_idx] = []
                # Первый список, который начал возвращать результаты
                if first_finished_list_idx is None:
                    first_finished_list_idx = list_idx
            
            list_results[list_idx].append(result)
        
        # 5. Сумма из первого завершившегося списка
        if first_finished_list_idx is not None:
            first_list_sum = sum(list_results[first_finished_list_idx])
            print(f"Сумма чисел в первом обработанном списке: {first_list_sum}")

if __name__ == "__main__":
    main()

Загружено списков: 10
Сумма чисел в первом обработанном списке: 11090
